In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [6]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [7]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [8]:
X


array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [77]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

In [11]:
model = Sequential()

In [16]:
model.add(Dense(32,activation = 'relu',input_dim = 8))
model.add(Dense(1,activation = 'sigmoid'))
model.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.fit(X_train,y_train,epochs = 100,batch_size = 32,validation_data=(X_test,y_test))

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6424 - loss: 0.6887 - val_accuracy: 0.6771 - val_loss: 0.6800
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6424 - loss: 0.6795 - val_accuracy: 0.6771 - val_loss: 0.6701
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6424 - loss: 0.6725 - val_accuracy: 0.6771 - val_loss: 0.6603
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6424 - loss: 0.6656 - val_accuracy: 0.6771 - val_loss: 0.6517
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6424 - loss: 0.6608 - val_accuracy: 0.6771 - val_loss: 0.6439
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6424 - loss: 0.6562 - val_accuracy: 0.6771 - val_loss: 0.6402
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6424 - loss: 0.6543 - val_accuracy: 0.6771 - val_loss: 0.6350
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6424 - loss: 0.6520 - val_accuracy: 0.6771 

## How to Select Appropriate Optimizer

In [19]:
%pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.1 MB/s eta 0:00:00


In [20]:
import keras_tuner as kt

In [21]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32,activation = 'relu',input_dim = 8))
  model.add(Dense(1,activation= 'sigmoid'))
  optmizer = hp.Choice('optimizer',values = ["adam",'sgd','rmsprop','adadelta'])
  model.compile(optimizer = optmizer, loss = 'binary_crossentropy',metrics = ['accuracy'])
  return model

In [23]:
tuner = kt.RandomSearch(build_model,objective = 'val_accuracy',max_trials = 5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [24]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.59375

Best val_accuracy So Far: 0.7708333134651184
Total elapsed time: 00h 00m 12s


In [25]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [27]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.fit(X_train,y_train,epochs = 100,initial_epoch = 5,validation_data = (X_test,y_test),batch_size = 32)

Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7517 - loss: 0.5404 - val_accuracy: 0.7760 - val_loss: 0.5226
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7535 - loss: 0.5227 - val_accuracy: 0.7812 - val_loss: 0.5090
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7552 - loss: 0.5115 - val_accuracy: 0.7812 - val_loss: 0.4977
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7604 - loss: 0.5022 - val_accuracy: 0.7865 - val_loss: 0.4881
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7604 - loss: 0.4945 - val_accuracy: 0.7865 - val_loss: 0.4804
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7587 - loss: 0.4887 - val_accuracy: 0.7917 - val_loss: 0.4744
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7604 - loss: 0.4841 - val_accuracy: 0.7865 - val_loss: 0.4695
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.4798 - val_accuracy: 0.786

## No.of Nodes in a layer

In [34]:
def build_model(hp):
  model = Sequential()
  unit = hp.Int('unit',min_value = 8,max_value = 128,step = 8)
  model.add(Dense(units = unit,activation = 'relu',input_dim = 8))
  model.add(Dense(1,activation = 'sigmoid'))
  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics = ['accuracy'])
  return model

In [37]:
tuner = kt.RandomSearch(build_model,objective = "val_accuracy",max_trials= 5,directory = 'mydir')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7604166865348816

Best val_accuracy So Far: 0.7916666865348816
Total elapsed time: 00h 00m 14s


In [39]:
tuner.get_best_hyperparameters()[0].values

{'unit': 128}

In [48]:
model = tuner.get_best_models(num_models = 1)[0]
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
model.fit(X_train,y_train,epochs = 100,initial_epoch = 5,validation_data = (X_test,y_test),batch_size = 32)

Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7726 - loss: 0.4843 - val_accuracy: 0.7917 - val_loss: 0.4554
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7743 - loss: 0.4745 - val_accuracy: 0.7917 - val_loss: 0.4488
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7778 - loss: 0.4689 - val_accuracy: 0.8073 - val_loss: 0.4460
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7760 - loss: 0.4643 - val_accuracy: 0.8021 - val_loss: 0.4428
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7778 - loss: 0.4609 - val_accuracy: 0.7969 - val_loss: 0.4423
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7795 - loss: 0.4582 - val_accuracy: 0.8021 - val_loss: 0.4432
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7812 - loss: 0.4556 - val_accuracy: 0.8021 - val_loss: 0.4412
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7795 - loss: 0.4530 - val_accuracy: 0.807

## How to select no.of layers

In [49]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(72,activation = 'relu',input_dim = 8))
  for i in range(hp.Int('num_layers',min_value = 1,max_value = 10)):
    model.add(Dense(72,activation = 'relu'))
  model.add(Dense(1,activation = 'sigmoid'))
  model.compile(optimizer = 'rmsprop',loss = 'binary_crossentropy',metrics = ['accuracy'])
  return model

In [51]:
tuner = kt.RandomSearch(build_model,objective = 'val_accuracy',max_trials = 5,directory = 'mydirLayers',project_name = 'NumLayers')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [52]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.8177083134651184

Best val_accuracy So Far: 0.8177083134651184
Total elapsed time: 00h 00m 24s


In [58]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3}

In [59]:
model = tuner.get_best_models(num_models = 1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 72)             │           648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 72)             │         5,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 72)             │         5,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 72)             │         5,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            73 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,489 (64.41 KB)

 Trainable params: 16,489 (64.41 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
model.fit(X_train,y_train,epochs = 100,initial_epoch = 5,validation_data = (X_test,y_test),batch_size = 32)

Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7899 - loss: 0.4451 - val_accuracy: 0.8073 - val_loss: 0.4493
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7899 - loss: 0.4245 - val_accuracy: 0.8229 - val_loss: 0.4502
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8038 - loss: 0.4110 - val_accuracy: 0.7865 - val_loss: 0.4658
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8021 - loss: 0.4134 - val_accuracy: 0.7604 - val_loss: 0.4753
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8247 - loss: 0.3943 - val_accuracy: 0.7865 - val_loss: 0.4765
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8177 - loss: 0.3953 - val_accuracy: 0.7552 - val_loss: 0.4805
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8125 - loss: 0.3853 - val_accuracy: 0.7969 - val_loss: 0.4854
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8264 - loss: 0.3756 - val_accuracy: 0.796

## All in ALL one model

In [78]:
def build_model(hp):
  model = Sequential()
  counter = 0;
  for i in range(hp.Int('num_layers',min_value = 1,max_value = 10)):
    if ( counter == 0) :
      model.add(
        Dense(
            units=hp.Int('units' + str(i),
            min_value = 8,
            max_value = 128,
            step = 8),
            activation = hp.Choice('activation' + str(i),values = ['relu','tanh','sigmoid']),
            input_dim = 8
        )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i),values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
      model.add(
          Dense(
              units=hp.Int('units' + str(i),
              min_value = 8,
              max_value = 128,
              step = 8),
              activation = hp.Choice('activation' + str(i),values = ['relu','tanh','sigmoid'])
          )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i),values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    counter = counter + 1
  model.add(Dense(1,activation = 'sigmoid'))
  model.compile(optimizer = hp.Choice('optimizer',values = ['adam','sgd','rmsprop','adadelta']),loss = 'binary_crossentropy',metrics = ['accuracy'])
  return model

In [79]:
tuner = kt.RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials = 5,
    directory = 'mydirAll',
    project_name = 'All'
)

Reloading Tuner from mydirAll/All/tuner0.json


In [80]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

In [81]:
model = tuner.get_best_models(num_models = 1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 104)            │           936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 104)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         3,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,713 (18.41 KB)

 Trainable params: 4,713 (18.41 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
model.fit(X_train,y_train,epochs = 100,initial_epoch = 5,validation_data = (X_test,y_test),batch_size = 32)

Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.7118 - loss: 0.5340 - val_accuracy: 0.7917 - val_loss: 0.4598
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6997 - loss: 0.5229 - val_accuracy: 0.8177 - val_loss: 0.4587
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7448 - loss: 0.5367 - val_accuracy: 0.8073 - val_loss: 0.4584
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7361 - loss: 0.5256 - val_accuracy: 0.8177 - val_loss: 0.4570
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7639 - loss: 0.5115 - val_accuracy: 0.8073 - val_loss: 0.4536
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7378 - loss: 0.5153 - val_accuracy: 0.8073 - val_loss: 0.4576
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7639 - loss: 0.4964 - val_accuracy: 0.8073 - val_loss: 0.4544
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7587 - loss: 0.5013 - val_accuracy: 0.8